# 01 — Recolección de Datos

Este notebook descarga:
1. **Precios históricos** de 5–10 mercados de Polymarket via CLOB API (datos diarios).
2. **Noticias globales** de GDELT para keywords relacionadas con cada mercado.

Salidas:
- `data/raw/prices_<slug>.csv`  → serie diaria por mercado
- `data/raw/news_<slug>.csv`   → artículos GDELT por mercado
- `data/raw/markets_catalog.csv` → catálogo completo de mercados seleccionados

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path so `src` is importable
ROOT = Path("__file__").resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)
print("Raw data dir:", RAW_DIR)

Project root: C:\Users\oscar\repos\predikt
Raw data dir: C:\Users\oscar\repos\predikt\data\raw


In [2]:
import pandas as pd
from datetime import datetime, timezone, timedelta

from src.polymarket import PolymarketClient
from src.gdelt_news import GdeltNewsClient

## 1. Catálogo de mercados (Gamma API)

Listamos mercados **cerrados** con alto volumen y seleccionamos los 8 más grandes.
La API devuelve entre otros el campo `clobTokenIds` que necesitaremos para el CLOB API.

In [3]:
pm = PolymarketClient(request_delay=0.4)

# Fetch top closed markets by volume (min $50 000 traded).
# end_date_min='2025-01-01' ensures we only get markets that closed recently,
# because the CLOB API does not retain price history for older markets.
markets = pm.top_markets(n=10, min_volume=50_000, end_date_min='2025-01-01')
print(f"Mercados encontrados: {len(markets)}")
markets[["slug", "question", "volume", "startDate", "endDate"]].head(10)

Mercados encontrados: 10


,slug,question,volume,startDate,endDate
0,will-the-panthers-win-super-bowl-2025,Will the Panthers win Super Bowl 2025?,1.393149e+08,2024-07-09T14:59:57.650118Z,2025-02-09T12:00:00Z
1,will-the-raiders-win-super-bowl-2025,Will the Raiders win Super Bowl 2025?,1.239699e+08,2024-07-09T15:28:42.42275Z,2025-02-09T12:00:00Z
2,will-the-titans-win-super-bowl-2025,Will the Titans win Super Bowl 2025?,1.195008e+08,2024-07-09T17:58:45.714Z,2025-02-09T12:00:00Z
3,will-the-browns-win-super-bowl-2025,Will the Browns win Super Bowl 2025?,1.152576e+08,2024-07-09T15:04:22.106699Z,2025-02-09T12:00:00Z
4,will-the-giants-win-super-bowl-2025,Will the Giants win Super Bowl 2025?,1.078573e+08,2024-07-09T15:36:12.169062Z,2025-02-09T12:00:00Z
5,will-the-patriots-win-super-bowl-2025,Will the Patriots win Super Bowl 2025?,7.427577e+07,2024-07-09T16:03:13.039758Z,2025-02-09T12:00:00Z
6,will-biden-complete-his-term-as-president,Will Biden finish his term?,6.455217e+07,2024-06-28T21:55:53.464Z,2025-01-20T12:00:00Z
7,will-the-falcons-win-super-bowl-2025,Will the Falcons win Super Bowl 2025?,4.988979e+07,2024-07-09T14:55:40.277835Z,2025-02-09T12:00:00Z
8,will-the-cardinals-win-super-bowl-2025,Will the Cardinals win Super Bowl 2025?,4.322874e+07,2024-07-09T14:51:27.110595Z,2025-02-09T12:00:00Z
9,will-the-texans-win-super-bowl-2025,Will the Texans win Super Bowl 2025?,3.330068e+07,2024-08-08T16:14:18.765Z,2025-02-10T12:00:00Z


In [4]:
# Save catalog
catalog_path = RAW_DIR / "markets_catalog.csv"
markets.to_csv(catalog_path, index=False)
print(f"Catálogo guardado en {catalog_path}")

Catálogo guardado en C:\Users\oscar\repos\predikt\data\raw\markets_catalog.csv


## 2. Precios históricos (CLOB API)

Para cada mercado extraemos la serie de precios del outcome YES (índice 0 de `clobTokenIds`).
El precio oscila entre 0 y 1, representando la probabilidad implícita del mercado.

In [5]:
price_frames = {}

for _, row in markets.iterrows():
    slug = row["slug"]
    try:
        df = pm.fetch_market_prices(row, outcome_index=0)
        if df.empty:
            print(f"[SKIP] {slug} — sin datos de precio")
            continue
        out_path = RAW_DIR / f"prices_{slug}.csv"
        df.to_csv(out_path, index=False)
        price_frames[slug] = df
        print(f"[OK] {slug} — {len(df)} días | {df['date'].min().date()} → {df['date'].max().date()}")
    except Exception as e:
        print(f"[ERROR] {slug}: {e}")

print(f"\nTotal mercados con precios: {len(price_frames)}")

[OK] will-the-panthers-win-super-bowl-2025 — 160 días | 2024-07-10 → 2024-12-16
[OK] will-the-raiders-win-super-bowl-2025 — 144 días | 2024-07-10 → 2024-11-30
[OK] will-the-titans-win-super-bowl-2025 — 153 días | 2024-07-10 → 2024-12-09
[OK] will-the-browns-win-super-bowl-2025 — 153 días | 2024-07-10 → 2024-12-09
[OK] will-the-giants-win-super-bowl-2025 — 143 días | 2024-07-10 → 2024-11-29
[OK] will-the-patriots-win-super-bowl-2025 — 146 días | 2024-07-10 → 2024-12-02
[OK] will-biden-complete-his-term-as-president — 206 días | 2024-06-29 → 2025-01-20
[OK] will-the-falcons-win-super-bowl-2025 — 181 días | 2024-07-10 → 2025-01-06
[OK] will-the-cardinals-win-super-bowl-2025 — 167 días | 2024-07-10 → 2024-12-23
[OK] will-the-texans-win-super-bowl-2025 — 194 días | 2024-07-10 → 2025-01-19

Total mercados con precios: 10


## 3. Noticias (GDELT Doc 2.0 API)

Para cada mercado construimos una keyword de búsqueda basada en el título.
GDELT cubre los últimos ~90 días. Para mercados más antiguos usaremos BigQuery (ver README).

In [6]:
# Define date window: last 3 months from today
END_DATE = datetime.now(tz=timezone.utc)
START_DATE = END_DATE - timedelta(days=89)  # GDELT rolling window ~90 days

start_str = START_DATE.strftime("%Y-%m-%d")
end_str   = END_DATE.strftime("%Y-%m-%d")
print(f"Ventana de noticias: {start_str} → {end_str}")

Ventana de noticias: 2026-01-25 → 2026-04-24


In [7]:
def make_keyword(question: str, max_words: int = 4) -> str:
    """Extract the most informative words from a market question as search keyword."""
    # Remove question marks and common filler
    import re
    skip = {"will", "the", "a", "an", "in", "be", "to", "of", "or", "and", "for", "at", "is"}
    tokens = re.sub(r"[^a-zA-Z0-9 ]", "", question.lower()).split()
    tokens = [t for t in tokens if t not in skip and len(t) > 2]
    return " ".join(tokens[:max_words])


gdelt = GdeltNewsClient(request_delay=1.2)
news_frames = {}

for _, row in markets.iterrows():
    slug = row["slug"]
    keyword = make_keyword(row["question"])
    print(f"Buscando: '{keyword}' para [{slug}]")
    try:
        articles = gdelt.search_articles(keyword, start_str, end_str)
        out_path = RAW_DIR / f"news_{slug}.csv"
        articles.to_csv(out_path, index=False)
        news_frames[slug] = articles
        print(f"{len(articles)} artículos guardados en {out_path.name}")
    except Exception as e:
        print(f"ERROR: {e}")

Buscando: 'panthers win super bowl' para [will-the-panthers-win-super-bowl-2025]


No articles found for keyword='panthers win super bowl' between 2026-01-25 and 2026-04-24


0 artículos guardados en news_will-the-panthers-win-super-bowl-2025.csv
Buscando: 'raiders win super bowl' para [will-the-raiders-win-super-bowl-2025]


No articles found for keyword='raiders win super bowl' between 2026-01-25 and 2026-04-24


0 artículos guardados en news_will-the-raiders-win-super-bowl-2025.csv
Buscando: 'titans win super bowl' para [will-the-titans-win-super-bowl-2025]


...

0 artículos guardados en news_will-the-falcons-win-super-bowl-2025.csv
Buscando: 'cardinals win super bowl' para [will-the-cardinals-win-super-bowl-2025]


GDELT search failed for keyword='cardinals win super bowl': HTTPSConnectionPool(host='api.gdeltproject.org', port=443): Max retries exceeded with url: /api/v2/doc/doc?query=%22cardinals%20win%20super%20bowl%22%20&startdatetime=20260125000000&enddatetime=20260424000000&maxrecords=250&mode=artlist&format=json (Caused by ConnectTimeoutError(<HTTPSConnection(host='api.gdeltproject.org', port=443) at 0x226c9cec910>, 'Connection to api.gdeltproject.org timed out. (connect timeout=None)'))


0 artículos guardados en news_will-the-cardinals-win-super-bowl-2025.csv
Buscando: 'texans win super bowl' para [will-the-texans-win-super-bowl-2025]


GDELT search failed for keyword='texans win super bowl': 


0 artículos guardados en news_will-the-texans-win-super-bowl-2025.csv


## 4. Resumen de recolección

In [8]:
summary = []
for slug in price_frames:
    n_prices = len(price_frames[slug])
    n_articles = len(news_frames.get(slug, pd.DataFrame()))
    q = markets.loc[markets["slug"] == slug, "question"].values[0]
    summary.append({"slug": slug, "question": q[:60], "n_prices": n_prices, "n_articles": n_articles})

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

# Save summary
(RAW_DIR / "collection_summary.csv").write_text(summary_df.to_csv(index=False))
print("\nResumen guardado.")

                                     slug                                question  n_prices  n_articles
    will-the-panthers-win-super-bowl-2025  Will the Panthers win Super Bowl 2025?       160           0
     will-the-raiders-win-super-bowl-2025   Will the Raiders win Super Bowl 2025?       144           0
      will-the-titans-win-super-bowl-2025    Will the Titans win Super Bowl 2025?       153           0
      will-the-browns-win-super-bowl-2025    Will the Browns win Super Bowl 2025?       153           0
      will-the-giants-win-super-bowl-2025    Will the Giants win Super Bowl 2025?       143           0
    will-the-patriots-win-super-bowl-2025  Will the Patriots win Super Bowl 2025?       146          42
will-biden-complete-his-term-as-president             Will Biden finish his term?       206           0
     will-the-falcons-win-super-bowl-2025   Will the Falcons win Super Bowl 2025?       181           0
   will-the-cardinals-win-super-bowl-2025 Will the Cardinals win

Como se puede observar, la recolección de datos externos a los proporcionados por la API de Polymarket fue pobre, por lo que lo pospusimos para la siguiente entrega.